# 06a — Sensitivity Analysis: Stratified by Pre-Exposure Period Length

**Motivation:**  
In NB06, the pre-period for exposed users is truncated at their first anchor comment. Users who first commented in early September have almost no pre-period data, while November commenters have a fuller baseline. This creates a systematic difference within the exposed group:

- **Early commenters (Sep):** thin pre-period, likely higher baseline anxiety, more engaged users
- **Late commenters (Nov):** fuller pre-period, more stable baseline estimates, more typical users

PSM in NB06 treats these as one group, matching on a `pre_mh_score` estimated very differently across subpopulations. The null ATT could partially be an artifact of noisy baselines washing out a real signal.

**Approach:**  
Run three versions of the PSM + DiD pipeline:
1. Full sample (NB06 baseline replication)
2. Exposed users with pre-period span ≥7 days
3. Exposed users with pre-period span ≥14 days

For each version report: ATT, p-value, 95% CI, pre_mh_score variance, PSM balance (SMD), and dose-response.

**Inputs:** `panel_scores.parquet`, `post_level_scores.parquet`, `dose_exposure.parquet`

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SUBREDDIT = 'gradadmissions'  # change to

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed' / SUBREDDIT
FIG_DIR  = ROOT / 'figures'

panel    = pd.read_parquet(DATA_DIR / 'panel_scores.parquet')
post_lvl = pd.read_parquet(DATA_DIR / 'post_level_scores.parquet')
dose_df  = pd.read_parquet(DATA_DIR / 'dose_exposure.parquet')

print(f'Panel rows: {len(panel):,}  |  Exposed: {panel["exposed"].sum():,}  |  Unexposed: {(~panel["exposed"]).sum():,}')

## 1) Compute pre-period span per user

In [ ]:
pre = post_lvl[post_lvl['window'] == 'pre'].copy()
pre['created_dt'] = pd.to_datetime(pre['created_dt'])

span = (
    pre.groupby(['author', 'cycle'])
       .agg(pre_first=('created_dt', 'min'), pre_last=('created_dt', 'max'))
       .reset_index()
)
span['pre_span_days'] = (span['pre_last'] - span['pre_first']).dt.days

panel = panel.merge(span[['author', 'cycle', 'pre_span_days']], on=['author', 'cycle'], how='left')
panel['log1p_n_posts_pre'] = np.log1p(panel['pre_n_posts'])

exposed = panel[panel['exposed']]
print('Pre-period span distribution for exposed users:')
print(exposed['pre_span_days'].describe().round(1))
print()
print(f'Span == 0 days (single post):  {(exposed["pre_span_days"] == 0).sum():,} / {len(exposed):,} ({100*(exposed["pre_span_days"]==0).mean():.1f}%)')
print(f'Span <  7 days:                {(exposed["pre_span_days"] <  7).sum():,} / {len(exposed):,} ({100*(exposed["pre_span_days"]< 7).mean():.1f}%)')
print(f'Span < 14 days:                {(exposed["pre_span_days"] < 14).sum():,} / {len(exposed):,} ({100*(exposed["pre_span_days"]<14).mean():.1f}%)')

In [ ]:
# Pre-period span by month of first pre-activity — confirms systematic pattern
exposed_with_span = panel[panel['exposed']].copy()
pre_first = pre.groupby(['author','cycle'])['created_dt'].min().reset_index().rename(columns={'created_dt':'pre_first'})
exposed_with_span = exposed_with_span.merge(pre_first, on=['author','cycle'], how='left')
exposed_with_span['pre_first_month'] = exposed_with_span['pre_first'].dt.to_period('M')

print('Median pre-period span (days) by month of first pre-activity:')
print(exposed_with_span.groupby('pre_first_month')['pre_span_days'].agg(['median','count']).to_string())

## 2) PSM + DiD helper functions

In [ ]:
def run_psm_did(df, caliper=0.05):
    """
    1:1 nearest-neighbor PSM per cycle on (pre_mh_score, log1p_n_posts_pre),
    then pooled DiD regression with HC3 standard errors.
    Returns dict of summary statistics.
    """
    matched_rows = []
    cycle_stats  = []

    for cycle in sorted(df['cycle'].unique()):
        c = df[df['cycle'] == cycle].copy().reset_index(drop=True)
        feats = ['pre_mh_score', 'log1p_n_posts_pre']

        scaler = StandardScaler()
        X = scaler.fit_transform(c[feats].fillna(0))
        y = c['exposed'].astype(int).values

        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(X, y)
        c['pscore'] = lr.predict_proba(X)[:, 1]

        exp_c   = c[c['exposed']].reset_index(drop=True)
        unexp_c = c[~c['exposed']].reset_index(drop=True)
        if len(exp_c) < 5 or len(unexp_c) < 5:
            continue

        nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
        nn.fit(unexp_c[['pscore']])
        dists, idxs = nn.kneighbors(exp_c[['pscore']])

        n_pairs = 0
        for i, (d, idx) in enumerate(zip(dists.flatten(), idxs.flatten())):
            if d <= caliper:
                matched_rows.append(exp_c.iloc[i].to_dict())
                matched_rows.append(unexp_c.iloc[idx].to_dict())
                n_pairs += 1
        cycle_stats.append({'cycle': cycle, 'n_pairs': n_pairs,
                             'n_exposed': len(exp_c), 'n_unexposed': len(unexp_c)})

    if not matched_rows:
        return None

    matched = pd.DataFrame(matched_rows).drop_duplicates(subset=['author', 'cycle'])

    # SMD on pre_mh_score (key balance metric)
    exp_m   = matched[matched['exposed']]['pre_mh_score']
    unexp_m = matched[~matched['exposed']]['pre_mh_score']
    pooled_sd = np.sqrt((exp_m.var() + unexp_m.var()) / 2)
    smd = abs(exp_m.mean() - unexp_m.mean()) / pooled_sd if pooled_sd > 0 else np.nan

    # Long-format DiD
    rows = []
    for _, r in matched.iterrows():
        rows.append({'author': r['author'], 'cycle': r['cycle'],
                     'exposed': int(r['exposed']), 'period': 0,
                     'mh_score': r['pre_mh_score'],
                     'log1p_posts': np.log1p(r['pre_n_posts'])})
        rows.append({'author': r['author'], 'cycle': r['cycle'],
                     'exposed': int(r['exposed']), 'period': 1,
                     'mh_score': r['post_mh_score'],
                     'log1p_posts': np.log1p(r['post_n_posts'])})
    long = pd.DataFrame(rows)
    long['period_x_exposed'] = long['period'] * long['exposed']

    mod = smf.ols(
        'mh_score ~ period + exposed + period_x_exposed + log1p_posts',
        data=long
    ).fit(cov_type='HC3')

    return {
        'n_panel':    len(df),
        'n_exposed':  int(df['exposed'].sum()),
        'pre_var':    df.loc[df['exposed'], 'pre_mh_score'].var(),
        'n_matched':  len(matched),
        'smd':        smd,
        'att':        mod.params['period_x_exposed'],
        'pval':       mod.pvalues['period_x_exposed'],
        'ci_lo':      mod.conf_int().loc['period_x_exposed', 0],
        'ci_hi':      mod.conf_int().loc['period_x_exposed', 1],
        'cycle_stats': cycle_stats,
        'matched':    matched,
    }

In [ ]:
def run_dose_response(df_panel):
    """
    Dose-response DiD on post-level observations for exposed users.
    Treatment = log1p(n_anchor_comments). Mirrors NB06 specification.
    """
    keep = set(zip(df_panel[df_panel['exposed']]['author'],
                   df_panel[df_panel['exposed']]['cycle']))

    dose = dose_df.copy()
    dose['key'] = list(zip(dose['author'], dose['cycle']))
    dose = dose[dose['key'].isin(keep)]

    # Post-period post-level scores for exposed users with dose data
    post_exp = post_lvl[post_lvl['window'] == 'post'].copy()
    post_exp = post_exp.merge(dose[['author', 'cycle', 'log1p_n_anchor']],
                               on=['author', 'cycle'], how='inner')

    # Pre-period mean score as baseline
    pre_scores = (
        post_lvl[post_lvl['window'] == 'pre']
        .groupby(['author', 'cycle'])['mean_mh_score'].mean()
        .reset_index().rename(columns={'mean_mh_score': 'pre_score'})
    )
    post_exp = post_exp.merge(pre_scores, on=['author', 'cycle'], how='left')

    # Build pre/post long format (one pre row per post-period observation)
    rows = []
    for _, r in post_exp.iterrows():
        rows.append({'period': 0, 'mh_score': r.get('pre_score', np.nan),
                     'log1p_n_anchor': r['log1p_n_anchor'], 'cycle': r['cycle']})
        rows.append({'period': 1, 'mh_score': r['mean_mh_score'],
                     'log1p_n_anchor': r['log1p_n_anchor'], 'cycle': r['cycle']})
    long = pd.DataFrame(rows).dropna(subset=['mh_score'])
    long['period_x_dose'] = long['period'] * long['log1p_n_anchor']

    mod = smf.ols(
        'mh_score ~ period + log1p_n_anchor + period_x_dose + C(cycle)',
        data=long
    ).fit(cov_type='HC3')

    return {
        'coef': mod.params['period_x_dose'],
        'pval': mod.pvalues['period_x_dose'],
        'ci_lo': mod.conf_int().loc['period_x_dose', 0],
        'ci_hi': mod.conf_int().loc['period_x_dose', 1],
        'n_obs': int(mod.nobs),
    }

## 3) Run three versions

In [ ]:
samples = [
    ('Full sample (NB06 baseline)', panel.copy()),
    ('Pre-period >= 7 days',        panel[(~panel['exposed']) | (panel['pre_span_days'] >= 7)].copy()),
    ('Pre-period >= 14 days',       panel[(~panel['exposed']) | (panel['pre_span_days'] >= 14)].copy()),
]

results = {}
for label, df in samples:
    print(f'Running: {label} ...')
    r   = run_psm_did(df)
    dr  = run_dose_response(df)
    results[label] = {'did': r, 'dose': dr}
    print(f'  Done. Matched N={r["n_matched"]:,}, ATT={r["att"]:+.4f} (p={r["pval"]:.4f})')
print('\nAll done.')

## 4) Summary table

In [ ]:
rows = []
for label, res in results.items():
    r  = res['did']
    dr = res['dose']
    rows.append({
        'Sample':            label,
        'N panel':           r['n_panel'],
        'N exposed':         r['n_exposed'],
        'pre_mh variance':   round(r['pre_var'], 5),
        'N matched':         r['n_matched'],
        'SMD (pre_mh)':      round(r['smd'], 4),
        'ATT':               round(r['att'], 4),
        'p-value':           round(r['pval'], 4),
        '95% CI':            f"[{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]",
        'Dose coef':         round(dr['coef'], 4),
        'Dose p':            round(dr['pval'], 4),
    })

summary = pd.DataFrame(rows).set_index('Sample')
display(summary)

## 5) Interpretation

**What to look for:**

- **pre_mh variance** should drop as we filter to cleaner baselines — confirms noise reduction
- **ATT** should move toward significance if noise was diluting a real effect (noise dilution story)
- **SMD** may degrade as the exposed pool shrinks — watch for values >0.10 (poor balance)
- **Dose-response** should remain robust across all three versions if it reflects a real signal

**Interpretation guide:**
- If ATT increases monotonically (full → ≥7 → ≥14) and reaches significance: strong support for noise dilution, report as main sensitivity finding
- If ATT moves at ≥7 but collapses at ≥14: effect is concentrated in moderate pre-period users, interpret cautiously
- If ATT stays null across all versions: null is real and robust to measurement noise — actually strengthens the null finding
- If dose-response weakens with filtering: dose signal was also partially noise-driven
- If dose-response holds or strengthens: most robust finding in the paper

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

labels = list(results.keys())
atts   = [results[l]['did']['att']   for l in labels]
ci_los = [results[l]['did']['ci_lo'] for l in labels]
ci_his = [results[l]['did']['ci_hi'] for l in labels]
pvals  = [results[l]['did']['pval']  for l in labels]

colors = ['steelblue' if p > 0.05 else 'firebrick' for p in pvals]

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = range(len(labels))

for i, (att, lo, hi, col) in enumerate(zip(atts, ci_los, ci_his, colors)):
    ax.plot([lo, hi], [i, i], color=col, linewidth=2)
    ax.scatter(att, i, color=col, s=80, zorder=5)

ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels)
ax.set_xlabel('ATT (DiD coefficient on period × exposed)')
ax.set_title('Sensitivity to pre-period length restriction\n(blue = p>0.05, red = p≤0.05)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_sensitivity_pre_period.png', dpi=150, bbox_inches='tight')
plt.show()